# Hierarchical RAG
Here, we load all the RAG chunk JSON files and go through each one to find chunks where the type is "figure". For those, we send the figure’s caption or related content to the GPT-4.1-mini API to get a short summary of what the figure shows, things like the axes, comparisons, or main trends. Then we store that summary back into a new key called image_summary and save everything in the rag_chunks_image_summary folder. This way, all the figures are now represented as text, making it much easier to embed and search later without depending on image embedding models that usually fail to capture the meaning of complex research plots.

In [4]:
import json
from pathlib import Path
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from collections import defaultdict
import re
import hashlib, faiss
import numpy as np

DATA_DIR = Path("data/rag_chunks_image_summary_v2")
all_chunks = []

EMBED_MODEL_ID = "Alibaba-NLP/gte-large-en-v1.5"
BATCH = 64

embed_model = SentenceTransformer(EMBED_MODEL_ID, trust_remote_code=True)

/home/mmk2266/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
for file in DATA_DIR.glob("*.rag.chunks.json"):
    paper_id = file.stem.split(".")[0]   # e.g. "2001.08361"
    with open(file, "r", encoding="utf-8") as f:
        chunks = json.load(f)
    for ch in chunks:
        all_chunks.append({
            "paper_id": paper_id,
            "id": ch["id"],
            "type": ch["type"],
            "content": ch.get("content", ""),
            "page": ch.get("metadata", {}).get("page"),
            "section": ch.get("metadata", {}).get("section"),
        })

## Level-3 is just like Basic RAG

In [6]:
def safe_section_id(s):
    if s is None:
        s = "unknown"
    s = s.strip()
    s = re.sub(r"\s+", " ", s)
    h = hashlib.md5(s.encode("utf-8")).hexdigest()[:12]
    return h

level3 = [
    {
        "uid": f"{x['paper_id']}__{x['id']}",
        "paper_id": x["paper_id"],
        "section": x["section"],
        "section_id": safe_section_id(x["section"]),
        "text": x["content"]
    }
    for x in all_chunks
    if x["content"].strip()
]

with open("data/metadata/meta_l3_1000.json", "w") as f:
    json.dump(level3, f, indent=2)

In [7]:
len(all_chunks), len(level3)

(34057, 34057)

In [8]:
def embed_items(items):
    texts = [x["text"] for x in items]
    embs = embed_model.encode(texts, batch_size=BATCH, show_progress_bar=True, normalize_embeddings=True)
    return np.array(embs).astype("float32")

In [9]:
# secton level

from collections import defaultdict

section_map = defaultdict(list)

for x in all_chunks:
    page = x["page"]
    if page is None:
        continue
    key = (x["paper_id"], f"page_{page}")
    section_map[key].append(x["content"])


level2 = []

for (pid, page), texts in section_map.items():
    sec_id = page              # e.g. "page_1"
    merged = "\n".join(texts)[:4000]

    level2.append({
        "uid": f"{pid}__{sec_id}",
        "paper_id": pid,
        "section": sec_id,
        "section_id": sec_id,
        "text": merged
    })

with open("data/metadata/meta_l2_1000.json", "w") as f:
    json.dump(level2, f, indent=2)

In [10]:
# document level

paper_map = defaultdict(list)

for x in all_chunks:
    if x["section"] in ("Abstract", "Summary", "Introduction"):
        paper_map[x["paper_id"]].append(x["content"])

level1 = []

for pid, texts in paper_map.items():
    merged = "\n".join(texts)[:4000]
    level1.append({
        "uid": pid,
        "paper_id": pid,
        "text": merged
    })

In [11]:
len(all_chunks), len(level3), len(level2), len(level1)

(34057, 34057, 357, 7)

# Embed all levels

In [12]:
def embed_items(items):
    texts = [x["text"] for x in items]
    embs = embed_model.encode(texts, batch_size=BATCH, show_progress_bar=True, normalize_embeddings=True)
    return np.array(embs).astype("float32")

In [13]:
emb_l1 = embed_items(level1)
emb_l2 = embed_items(level2)
emb_l3 = embed_items(level3)

Batches: 100%|██████████| 533/533 [38:20<00:00,  4.32s/it] 


In [14]:
# save embeddings

np.save("data/embeddings/emb_l1_1000.npy", emb_l1)
np.save("data/embeddings/emb_l2_1000.npy", emb_l2)
np.save("data/embeddings/emb_l3_1000.npy", emb_l3)

# Build 3 FAISS indices

In [15]:
FAISS_DIR = Path("data/faiss")
FAISS_DIR.mkdir(exist_ok=True, parents=True)

In [18]:
emb_l1 = np.load("data/embeddings/emb_l1_1000.npy")
emb_l2 = np.load("data/embeddings/emb_l2_1000.npy")
emb_l3 = np.load("data/embeddings/emb_l3_1000.npy")

d = emb_l1.shape[1]

def build_hnsw(embs):
    idx = faiss.IndexHNSWFlat(d, 32)
    idx.hnsw.efConstruction = 200
    idx.add(embs)
    return idx

# idx_l1 = build_hnsw_index(emb_l1)
# idx_l2 = build_hnsw_index(emb_l2)
# idx_l3 = build_hnsw_index(emb_l3)

In [19]:
# --------------------------------------------------
# 2. LEVEL-1: global paper index
# --------------------------------------------------

idx_l1 = build_hnsw(emb_l1)
faiss.write_index(idx_l1, str(FAISS_DIR / "l1_global.faiss"))

with open("data/metadata/meta_l1_1000.json", "w") as f:
    json.dump(level1, f, indent=2)

In [20]:
# --------------------------------------------------
# 3. LEVEL-2: per-paper page indices
# --------------------------------------------------

l2_by_paper = defaultdict(list)

for i, x in enumerate(level2):
    l2_by_paper[x["paper_id"]].append(i)

l2_manifest = {}

for paper_id, idxs in l2_by_paper.items():
    sub_embs = emb_l2[idxs]
    sub_index = build_hnsw(sub_embs)

    out_path = FAISS_DIR / f"l2_{paper_id}.faiss"
    faiss.write_index(sub_index, str(out_path))

    # store mapping so query-time lookup is trivial
    l2_manifest[paper_id] = {
        "index_path": str(out_path),
        "meta_indices": idxs
    }

with open("data/metadata/l2_manifest.json", "w") as f:
    json.dump(l2_manifest, f, indent=2)



In [21]:
# --------------------------------------------------
# 4. LEVEL-3: per-(paper,page) chunk indices
# --------------------------------------------------
import re
import hashlib



l3_by_key = defaultdict(list)

for i, x in enumerate(level3):
    safe_section = safe_section_id(x["section"])
    key = f"{x['paper_id']}__{safe_section}"   # ← USE SAFE NAME
    l3_by_key[key].append(i)


l3_manifest = {}



for key, idxs in l3_by_key.items():
    sub_embs = emb_l3[idxs]
    sub_index = build_hnsw(sub_embs)

    out_path = FAISS_DIR / f"l3_{key}.faiss"
    faiss.write_index(sub_index, str(out_path))

    paper_id, sec_id = key.split("__", 1)

    l3_manifest[key] = {
    "index_path": str(out_path),
    "meta_indices": idxs,
    "paper_id": x["paper_id"],
    "section_id": sec_id
    }

with open("data/metadata/l3_manifest.json", "w") as f:
    json.dump(l3_manifest, f, indent=2)



In [23]:
print("len(level3):", len(level3))
print("emb_l3 shape:", emb_l3.shape)
print("max index used:", max(i for idxs in l3_by_key.values() for i in idxs))

len(level3): 34057
emb_l3 shape: (34057, 1024)
max index used: 34056


# Adaptive Depth-Controller

In [24]:
def embed_query(q):
    return embed_model.encode([q], normalize_embeddings=True).astype("float32")


def choose_depth(query):
    q = query.lower()
    if re.search(r"(figure|eqn|equation|derive|value|algorithm)", q):
        return 3
    if len(query.split()) <= 6:
        return 1
    if len(query.split()) <= 15:
        return 2
    return 3

In [25]:
def search_spi(query, k1=5, k2=10, k3=20):
    depth = choose_depth(query)
    qemb = embed_query(query)

    # ---- LEVEL 1 ----
    D1, I1 = idx_l1.search(qemb, k1)
    papers = [level1[i] for i in I1[0]]

    if depth == 1:
        return {"level": 1, "papers": papers}

    # ---- LEVEL 2 ----
    cand_pids = {p["paper_id"] for p in papers}
    l2_filtered = [i for i, x in enumerate(level2) if x["paper_id"] in cand_pids]
    if not l2_filtered:
        return {"level": 1, "papers": papers}

    # TEMP FAISS sub-index for level 2
    dim2 = emb_l2.shape[1]
    sub2 = faiss.IndexFlatIP(dim2)
    sub2.add(emb_l2[l2_filtered])

    D2, I2 = sub2.search(qemb, k2)
    sections = [level2[l2_filtered[i]] for i in I2[0]]

    if depth == 2:
        return {"level": 2, "papers": papers, "sections": sections}

    # ---- LEVEL 3 ----
    cand_sections = {(s["paper_id"], s["section"]) for s in sections}
    l3_filtered = [i for i, x in enumerate(level3)
                   if (x["paper_id"], x["section"]) in cand_sections]

    if not l3_filtered:
        return {"level": 2, "papers": papers, "sections": sections}

    # TEMP FAISS sub-index for level 3
    dim3 = emb_l3.shape[1]
    sub3 = faiss.IndexFlatIP(dim3)
    sub3.add(emb_l3[l3_filtered])

    D3, I3 = sub3.search(qemb, k3)
    chunks = [level3[l3_filtered[i]] for i in I3[0]]

    return {"level": 3, "papers": papers, "sections": sections, "chunks": chunks}


In [26]:
query = "What is the scaling law relationship between compute and test loss?"
result = search_spi(query)

print("=== Retrieval Level Chosen ===")
print(result["level"])

=== Retrieval Level Chosen ===
2


In [27]:
print("\n=== PAPERS (Level 1) ===")
for p in result.get("papers", []):
    print(f"[{p['paper_id']}]")
    print(p['text'], "...\n")


=== PAPERS (Level 1) ===
[2001]
Language provides a natural domain for the study of artificial intelligence, as the vast majority of reasoning tasks can be efficiently expressed and evaluated in language, and the world's text provides a wealth of data for unsupervised learning via generative modeling. Deep learning has recently seen rapid progress in language modeling, with state of the art models [RNSS18, DCLT18, YDY + 19, LOG + 19, RSR + 19] approaching human-level performance on many specific tasks [WPN + 19], including the composition of coherent multiparagraph prompted text samples [RWC + 19]. One might expect language modeling performance to depend on model architecture, the size of neural models, the computing power used to train them, and the data available for this training process. In this work we will empirically investigate the dependence of language modeling loss on all of these factors, focusing on the Transformer architecture [VSP + 17, LSP + 18]. The high ceiling and l

In [28]:
print("\n=== SECTIONS (Level 2) ===")
for s in result.get("sections", []):
    print(f"[{s['paper_id']} - {s['section']}]")
    print(s['text'], "...\n")


=== SECTIONS (Level 2) ===
[2001 - page_15]
Figure 12 Left: Given a fixed compute budget, a particular model size is optimal, though somewhat larger or smaller models can be trained with minimal additional compute. Right: Models larger than the computeefficient size require fewer steps to train, allowing for potentially faster training if sufficient additional parallelism is possible. Note that this equation should not be trusted for very large models, as it is only valid in the power-law region of the learning curve, after initial transient effects.
Figure 13 When adjusting performance to simulate training far below the critical batch size, we find a somewhat altered power law for L ( C min ) when compared with the fully empirical results. The conspicuous lump at 10 -5 PF-days marks the transition from 1-layer to 2-layer networks; we exclude 1-layer networks in the power-law fits. It is the L ( C min ) trend that we expect to provide a reliable extrapolation for larger compute.
that 

In [29]:
print("\n=== CHUNKS (Level 3) ===")
for c in result.get("chunks", []):
    print(f"[{c['uid']}]")
    print(c['text'][:250], "...\n")


=== CHUNKS (Level 3) ===


# Generation

In [30]:
SYSTEM = (
  "Answer ONLY from <chunk> context. Read EVERY chunk.\n"
  "Step 1: Extract EVERY distinct method/model by its PROPER NAME as written "
  "(use exact capitalization), then give a 1-line description.\n"
  "Step 2: Write a concise answer that covers each named item.\n"
  "If nothing relevant: Not found in the given context.\n"
  "Paraphrase descriptions; DO NOT rename methods. Cite as [DOC:doc_id, p:page]."
)

GEN_CFG = dict(
    max_new_tokens=512,        # maximum tokens model can generate in the output
    temperature=0.5,           # randomness; lower = more deterministic, higher = more creative
    top_p=0.9,                 # nucleus sampling; model samples only from top 90% probability mass
    do_sample=False,            # if True, enables stochastic sampling (else it picks argmax each time)
    repetition_penalty=1.01,   # penalizes repeating same tokens; >1 discourages loops/redundancy
    no_repeat_ngram_size=8,    # forbids repeating any 8-token sequence exactly
)

In [31]:
def build_context(result, max_chunks=10):
    chunks = result.get("chunks", [])[:max_chunks]
    sections = result.get("sections", [])[:3]  # optional

    context_blocks = []

    # Add sections (higher-level summaries)
    for sec in sections:
        context_blocks.append(f"<chunk id='{sec['uid']}'>\n{sec['text']}\n</chunk>")

    # Add fine chunks
    for ch in chunks:
        context_blocks.append(f"<chunk id='{ch['uid']}'>\n{ch['text']}\n</chunk>")

    return "\n\n".join(context_blocks)

In [32]:
def build_prompt(query, context):
    return f"{SYSTEM}\n\n<context>\n{context}\n</context>\n\nUser question: {query}\nAnswer:"


In [33]:
import torch, faiss, ujson as json
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

GEN_MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
tok = AutoTokenizer.from_pretrained(GEN_MODEL_ID)

bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)

generator_model = AutoModelForCausalLM.from_pretrained(
    GEN_MODEL_ID,
    quantization_config=bnb_cfg,
    device_map="auto",
    low_cpu_mem_usage=True,
)

Loading checkpoint shards: 100%|██████████| 4/4 [01:27<00:00, 21.79s/it]


In [34]:
def generate_answer(prompt):
    inputs = tok(prompt, return_tensors="pt").to(generator_model.device)
    output = generator_model.generate(**inputs, **GEN_CFG)
    return tok.decode(output[0], skip_special_tokens=True)

In [35]:
def answer_question(query, k1=5, k2=10, k3=20):
    # retrieval
    result = search_spi(query, k1=k1, k2=k2, k3=k3)

    # build context
    context = build_context(result)

    # build prompt
    prompt = build_prompt(query, context)

    # generate
    answer = generate_answer(prompt)

    return {
        "retrieval_level": result["level"],
        "context": context,
        "prompt": prompt,
        "answer": answer
    }

In [36]:
resp = answer_question("What methods exist to accelerate token generation during inference without retraining the language model")
print(resp["answer"])

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Answer ONLY from <chunk> context. Read EVERY chunk.
Step 1: Extract EVERY distinct method/model by its PROPER NAME as written (use exact capitalization), then give a 1-line description.
Step 2: Write a concise answer that covers each named item.
If nothing relevant: Not found in the given context.
Paraphrase descriptions; DO NOT rename methods. Cite as [DOC:doc_id, p:page].

<context>
<chunk id='2001__page_7'>
Table 1 Parameter counts and compute (forward pass) estimates for a Transformer model. Sub-leading terms such as nonlinearities, biases, and layer normalization are omitted.
For contexts and models with d model > n ctx / 12, the context-dependent computational cost per token is a relatively small fraction of the total compute. Since we primarily study models where d model glyph[greatermuch] n ctx / 12, we do not include context-dependent terms in our training compute estimate. Accounting for the backwards pass (approximately twice the compute as the forwards pass), we then define